In [1]:
import os, sys, re, h5py
import numpy as np
from lib import NanonisAPI, MLAAPI
from lib import FileFunctions, DataProcessing
import time
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import sklearn.gaussian_process as gp
from sklearn.model_selection import train_test_split
from scipy.signal import fftconvolve

data = DataProcessing()
file_functions = FileFunctions()
(hw_config, error) = file_functions.load_yaml("./sys/config.yml")

#nanonis = NanonisAPI(hw_config = hw_config, message_callback = lambda message, message_type: print(f"{message} [{message_type}]"), status_callback = lambda status_str: print(f"Status: {status_str}"))
#mla = MLAAPI(hw_config = hw_config, message_callback = lambda message, message_type: print(f"{message} [{message_type}]"), status_callback = lambda status_str: print(f"Status: {status_str}"))
#nhw = nanonis.nanonis_hardware
#nanonis.link()

In [38]:
start_time = time.time()

for _ in range(1000):
    (header, end_pos) = file_functions.get_raw_sxm_header_new(file_path)
    header_array = np.array(header.split())

    nanonis_tags = [b":SCAN_RANGE:", b":SCAN_ANGLE:", b":SCAN_OFFSET:", b":REC_TIME:", b":REC_DATE:", b":SCAN_PIXELS:", b":SCAN_DIR:", b":BIAS:"]
    sct_tags = ["scan_range (m)", "angle (deg)", "offset (m)", "start_time", "scan_date", "grid_size", "up_or_down", "V_nanonis (V)"]
    value_lengths = [2, 1, 2, 1, 1, 2, 1, 1]

    sct_dict = {}
    for nanonis_tag, sct_tag, value_length in zip(nanonis_tags, sct_tags, value_lengths):
        try:
            index = np.where(header_array == nanonis_tag)[0][0]
            values_b = header_array[index + 1 : index + 1 + value_length]
            values_num = [file_functions.get_scientific_numbers(value.decode("utf-8"))[0] for value in values_b]
            if len(values_num) < 2: values_num = values_num[0]
            sct_dict.update({sct_tag: values_num})
        except:
            pass

end_time = time.time()

print(f"{end_time - start_time = :.4f}")
print(f"{sct_dict = }")

[pixels, lines] = [int(sct_dict.get("grid_size", [1, 1])[i]) for i in range(2)]
sct_dict.update({"pixels": pixels, "lines": lines})
frame = file_functions.convert_to_sct_frame(sct_dict)
sct_dict.update({"grid": frame})

end_time - start_time = 0.1146
sct_dict = {'scan_range (m)': [2.2e-07, 6e-08], 'angle (deg)': 20.0, 'offset (m)': [2e-07, 4e-07], 'start_time': 9.0, 'scan_date': 10.06, 'grid_size': [48.0, 96.0], 'V_nanonis (V)': 1.1}


In [ ]:
def minimal_sxm_header_read(file_path):
    try:
        (raw_header, end_pos) = file_functions.get_raw_sxm_header(file_path)
        header = np.array(raw_header, dtype = np.str_)

        nanonis_tags = [":SCAN_RANGE:\n", ":SCAN_ANGLE:\n", ":SCAN_OFFSET:\n", ":REC_TIME:\n", ":REC_DATE:\n", ":SCAN_PIXELS:\n", ":SCAN_DIR:\n", ":BIAS:\n"]
        sct_tags = ["scan_range (m)", "angle (deg)", "offset (m)", "start_time", "scan_date", "grid_size", "up_or_down", "V_nanonis (V)"]

        sct_dict = {}
        for nanonis_tag, sct_tag in zip(nanonis_tags, sct_tags):
            try:
                index = np.where(header == nanonis_tag)[0][0]
                values_split = header[index + 1].split()

                if nanonis_tag in [":REC_TIME:\n", ":REC_DATE:\n", ":SCAN_DIR:\n"]:
                    sct_dict.update({sct_tag: values_split[0]})
                else:
                    values_num = [file_functions.get_scientific_numbers(value)[0] for value in values_split]
                    if len(values_num) < 2: values_num = values_num[0]
                    sct_dict.update({sct_tag: values_num})
            except Exception as e:
                print(f"Problem reading tag {nanonis_tag.split()[0]} from .sxm file")
    except Exception as e:
        print(f"Error getting the SXM file header: {e}")
    return (header, sct_dict)

def full_sxm_header_read(file_path):
    (header_array, sct_dict) = minimal_sxm_header_read(file_path)
    [pixels, lines] = [int(sct_dict.get("grid_size", [1, 1])[i]) for i in range(2)]
    sct_dict.update({"pixels": pixels, "lines": lines})
    frame = file_functions.convert_to_sct_frame(sct_dict)
    sct_dict.update({"frame": frame})

    # Z_controller
    try:
        z_controller_index = np.where(header_array == ":Z-CONTROLLER:\n")[0][0]
        z_controller_data = header_array[z_controller_index + 2].split("\t")
        [_, z_controller_name, feedback, setpoint, p_gain, i_gain, t_const] = z_controller_data
        sct_dict.update({"z_controller": {"name": z_controller_name, "feedback": bool(feedback), "setpoint": setpoint, "p_gain": p_gain, "i_gain": i_gain, "t_const": t_const}})
    except Exception as e:
        print(f"Problem retrieving z-controller data from .sxm file: {e}")

    # Channels
    try:
        channel_data_index = np.where(header_array == ":DATA_INFO:\n")[0][0]
        channel_names = []
        for channel_index in range(100):
            channel_data = header_array[channel_data_index + 2 + channel_index].split()
            if len(channel_data) < 1: break
            [channel_index, quantity, unit, direction, calibration, offset] = channel_data
            channel_names.append(" ".join((quantity, f"({unit})")))
        
        sct_dict.update({"channels": channel_names})
    except Exception as e:
        print(f"Problem retrieving channel data from .sxm file: {e}")
    return (header_array, sct_dict)


end - mid = 0.0002484321594238281, mid - start = 0.0003635883331298828


In [49]:
header_array

array([':NANONIS_VERSION:\n', '2\n', ':SCANIT_TYPE:\n',
       '              FLOAT            MSBFIRST\n', ':REC_DATE:\n',
       ' 10.06.2026\n', ':REC_TIME:\n', '09:50:10\n', ':REC_TEMP:\n',
       '      290.0000000000\n', ':ACQ_TIME:\n', '        12.4\n',
       ':SCAN_PIXELS:\n', '       48       96\n', ':SCAN_FILE:\n',
       'C:\\Nanonis\\2026-06-10\\unnamed0001.sxm\n', ':SCAN_TIME:\n',
       '             1.584E-1             1.584E-1\n', ':SCAN_RANGE:\n',
       '           2.200000E-7           6.000000E-8\n',
       ':SCAN_OFFSET:\n',
       '             2.000000E-7         4.000000E-7\n', ':SCAN_ANGLE:\n',
       '            2.000E+1\n', ':SCAN_DIR:\n', 'down\n', ':BIAS:\n',
       '1.100000E+0\n', ':Z-CONTROLLER:\n',
       '\tName\ton\tSetpoint\tP-gain\tI-gain\tT-const\n',
       '\tlog Current\t1\t5.000E-11 A\t2.000E-10 m\t2.000E-7 m/s\t1.000E-3 s\n',
       ':COMMENT:\n', '\n', ':DATA_INFO:\n',
       '\tChannel\tName\tUnit\tDirection\tCalibration\tOffset\n',
      

In [ ]:
z_imgs = [scans[i][0, 2] for i in range(2)]
scan_range_nm = grids[0].get("scan_range (nm)")
[w_nm, h_nm] = scan_range_nm
[pix, lines] = z_imgs[0].shape
width_per_pix_nm = w_nm / pix
height_per_line_nm = h_nm / lines

imgs_norm = [z_imgs[i] - np.mean(z_imgs[i]) for i in range(2)]

img_correlation = fftconvolve(imgs_norm[0], imgs_norm[1][::-1, ::-1], mode = "same")
y_peak, x_peak = np.unravel_index(np.argmax(img_correlation), img_correlation.shape)

y_center, x_center = img_correlation.shape[0] // 2, img_correlation.shape[1] // 2
y_shift_nm = (y_peak - y_center) * height_per_line_nm
x_shift_nm = (x_peak - x_center) * width_per_pix_nm

print(f"Detected Shift (Y, X): ({y_shift_nm}, {x_shift_nm})")



fig, ax = plt.subplots(1, 3, figsize = (10, 5))
ax[0].imshow(np.hstack((z_imgs[0], z_imgs[1], img_correlation / 1000)))
ax[1].imshow(z_imgs[1])
ax[2].imshow(img_correlation)
plt.show()

In [ ]:
file_path = "C:\\Nanonis\\2026-05-29\\scan_052.hdf5"
with h5py.File(file_path, "r") as f:
    grid_attrs = f["grid"].attrs
    grid = {key: grid_attrs[key] for key in grid_attrs.keys()}
    scan = f["scan_group"]["scan"][:]
z_fwd_img = scan[0, 2]

[w_nm, h_nm] = grid["scan_range (nm)"]
[n_dirs, n_channels, pixels, lines] = scan.shape
w_pix = w_nm / pixels
h_pix = h_nm / lines

x_linspace = np.linspace(w_pix / 2 - w_nm / 2, w_nm / 2 - w_pix /2, pixels)
y_linspace = np.linspace(h_pix / 2 - h_nm / 2, h_nm / 2 - h_pix /2, lines)
x_grid, y_grid = np.meshgrid(x_linspace, y_linspace)
x_list = x_grid.flatten()
y_list = y_grid.flatten()
list_indices = np.arange(len(x_list))
xy_all_coords_nm = np.array([[x_list[index], y_list[index]] for index in list_indices])

train_indices, test_indices = train_test_split(list_indices, test_size = .999)
grid_train_indices = np.array([[index % pixels, int(index / pixels)] for index in train_indices])
xy_train_coords_nm = np.array([[x_grid[pixel, line], y_grid[pixel, line]] for pixel, line in grid_train_indices])
z_train_coords_nm = np.array([z_fwd_img[pixel, line] for pixel, line in grid_train_indices])

k_fine_detail = gp.kernels.Matern(length_scale = .3, length_scale_bounds = (.1, 1.0), nu = 1.5)
#k_intermediate = gp.kernels.Matern(length_scale = 4, length_scale_bounds = (1.5, 8.0), nu = 1.5)
k_large_scale = gp.kernels.Matern(length_scale = 20, length_scale_bounds = (5, 100), nu = 2.5)
white_kernel = gp.kernels.WhiteKernel(noise_level = .01, noise_level_bounds = (1e-6, .2))
composite_kernel = k_fine_detail + k_large_scale

print("fitting")
gpr = gp.GaussianProcessRegressor(composite_kernel, normalize_y = True)
gpr.fit(xy_train_coords_nm, z_train_coords_nm)

print("extrapolating")
(z_fit_col, z_std_dev_col) = gpr.predict(xy_all_coords_nm, return_std = True)
z_fit_nm = z_fit_col.reshape(x_grid.shape)
z_std_dev_nm = z_std_dev_col.reshape(x_grid.shape)

print("plotting")
fig, ax = plt.subplots(1, 3)
ax[0].imshow(z_fwd_img)
ax[1].imshow(z_fit_nm)
ax[2].imshow(z_std_dev_nm)
plt.show()

In [ ]:
z_std_dev_nm_masked = z_std_dev_col.copy()

fine_length_scale2 = (gpr.kernel_.get_params().get("k1__length_scale", .5)) ** 2
selected_indices = []

for point in range(10):
    best_index = np.argmax(z_std_dev_nm_masked)
    selected_indices.append(best_index)
    next_coord = xy_all_coords_nm[best_index]
    
    # Mask the region of the selected point
    for index, xy in enumerate(xy_all_coords_nm):
        if np.sum((xy - next_coord) ** 2, axis = -1) < fine_length_scale2: z_std_dev_nm_masked[index] = 0

z_std_dev_masked = z_std_dev_nm_masked.reshape(x_grid.shape)
plt.imshow(z_std_dev_masked)
plt.plot()


In [ ]:
x_list = np.linspace(-10, 10, 101, dtype = np.float32)
mask = np.ones_like(x_list, dtype = np.float32)

for index, x in enumerate(x_list):
    mask[index] *= 1 - 1 / (1 + np.sum((x - 1.5) ** 2, axis = -1))

plt.plot(x_list, mask)
plt.show()

In [19]:
xy_coords = np.column_stack((x.reshape(-1), y.reshape(-1)))
z_col = z.reshape(-1, 1)

guess_l = (2., 1.)  # In general, x and y have different scales
bounds_l = ((1e-1, 100.),) * 2  # Same bounds for x and y
guess_n = 1.  # Amount of noise
bounds_n = (1e-20, 10.) # Bounds for noise
kernel = gp.kernels.RBF()

xy_train, xy_test, z_train, z_test = train_test_split(xy_coords, z_col, test_size = 0.95)

gpr = gp.GaussianProcessRegressor(kernel, normalize_y = True)


In [26]:
xy_coords.shape

(400, 2)

In [22]:
(z_fit_col, z_std_dev_col) = gpr.predict(xy_coords, return_std = True)
z_fit = z_fit_col.reshape(x.shape)
z_std_dev = z_std_dev_col.reshape(x.shape)

In [2]:
(grid, error) = nanonis.grid_update()
(lists, error) = nanonis.grids_to_lists(grid, direction = "down")

[x_list, y_list] = [lists.get(key) for key in ["x_list (nm)", "y_list (nm)"]]
list_len = len(x_list)

rng = np.random.default_rng()
random_flat_indices = rng.choice(a = list_len, size = 20, replace = False)
random_coordinates = np.array([[x_list[index], y_list[index]] for index in random_flat_indices])

Status: running
